# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library, following a step-by-step workflow.

### Dataset Source
The dataset source is provided via a Croissant schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment if running in Colab or a new environment)
!pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Let's review the available record sets and fields in the dataset, with all entities referenced by their `@id` fields, as required by Croissant.

In [ ]:
# List all record sets by @id and name, and display their fields with @id and name
print('\nAvailable Record Sets:')
recordsets = list(metadata.record_sets)
for rs in recordsets:
    print(f"- @id: {rs.id} | name: {rs.name}")
    print('  Fields:')
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name} | data_type: {field.data_type}")
    print()

## 3. Data Extraction
Let's load data from the primary record set containing the main clinical and pathological records. We'll identify the correct record set `@id` from above, and use that (as well as field `@id`s) for extraction and analysis. If there are multiple record sets, we will extract all of them.

In [ ]:
# Prepare to extract dataframes for each record set by @id
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set '@id': {record_set_id} — shape: {df.shape}")
    print(f"Columns: {list(df.columns)}\n")

## 4. Exploratory Data Analysis (EDA)
We'll now demonstrate basic analysis using the main clinical record set. We'll:
- Select a numeric field (e.g., age) using its `@id`
- Filter records based on this value
- Normalize this numeric field
- Group data by a key attribute (such as a tumor location or molecular status)

Please check the column names above and adjust the field `@id` as needed.

In [ ]:
# Pick the main record set for EDA:
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]
# Print available columns for convenience
print(f"Available columns in main record set ({main_record_set_id}):\n{df.columns.tolist()}")

# Choose a numeric field by its @id, e.g., 'age' (replace if the column is named differently, use the id from the schema, e.g. 'http://senscience.ai/age')
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise ValueError("Could not identify a column containing 'age'. Please review columns above.")
print(f"Numeric field selected: {numeric_field_id}")

# Filter records with age > 60
threshold = 60
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
else:
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold} (N={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') -
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a grouping field by @id, try MSI status or anatomical site, fallback to the first non-numeric column
group_field_id = None
for col in df.columns:
    if ('msi' in col.lower() or 'status' in col.lower() or 'anatomical' in col.lower() or 'site' in col.lower()) and col != numeric_field_id:
        group_field_id = col
        break
if not group_field_id:
    # Fallback: take the first string column different from numeric_field_id
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (e.g. age) and relationship with a grouping field (such as MSI-status or anatomical site).

We'll use matplotlib and seaborn for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style and increase figure size
sns.set(style="whitegrid")
plt.figure(figsize=(10, 5))

# Distribution of numeric variable (age)
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field available, plot boxplot of numeric variable by grouping field
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load and examine the FAIR² clinical dataset using Croissant's schema
- Reference all dataset elements (record sets, fields) by their `@id`
- Extract specific columns and perform basic EDA and visualization

This workflow can be adapted for detailed domain-specific analysis, model development, or clinical stratification using any dataset that provides a Croissant schema.

> **Note:** Always refer to the data dictionary or documentation (where available) for correct field meanings and usage.